# The Fiscal Amplifier: M2-to-Inflation Passthrough Under High Debt

In [1]:
import datetime

# ===== GLOBAL VARIABLES =====
RELOAD_DATA = False 
TODAY = str(datetime.datetime.now().date())
DEBT_TO_GDP_PERCENTILE = 67
# ============================

### Do Drivers of Inflation Change under different Debt Regimes? 

i.e. expectations are regime-dependent based on this equation:

$$\text{CPI\_YoY} = \beta_0 + \beta_1(\text{M2\_YoY\_Lag78w}) + \beta_2(\text{CPI\_Energy\_YoY}) + \beta_3(\text{Inflation\_Momentum}) + \beta_4(\text{Expect\_Anchor\_Signed}) + \varepsilon$$

#### Get Most Recent Data

In [2]:
# ===== IMPORTS =============
from otter import Pond
from otter.simulation import *
import otter.transforms as tf
from loader import load_fred_master
from macro_scores import score
import statsmodels.api as sm
# ============================

# ====== INIT ================
if RELOAD_DATA:
    _ = load_fred_master()
fred = Pond("data/fred_master.csv", handler=score)
print(f"Today is {TODAY}.")
fred.data = fred.data[fred.data["date"]<=TODAY]
fred.data.tail()
# ============================

Today is 2026-03-28.


,date,Headline_CPI,Core_CPI,PCE_Price_Index,Core_PCE,CPI_Food,CPI_Energy,UMich_Inflation_Expectations,Breakeven_Inflation_5Y,Breakeven_Inflation_10Y,...,Curve_10Y2Y_Lag_52w,Curve_10Y3M_Lag_52w,Breakeven_5Y_Lag_26w,Breakeven_5Y_Lag_52w,Home_Price_YoY_Lag_78w,Credit_Impulse_Lag_13w,Credit_Impulse_Lag_26w,FFR_Chg_52w,FFR_Chg_52w_Lag_52w,Sentiment_Lag_13w
1886,2026-02-27,327.46,333.512,128.969,128.394,346.622,283.223,3.4,2.410,2.266,...,0.234,-1.600000e-02,2.490,2.574,3.544575,1.339104,-6.694157,-0.69,-1.0,51.0
1887,2026-03-06,327.46,333.512,128.969,128.394,346.622,283.223,3.4,2.490,2.306,...,0.282,-9.000000e-02,2.450,2.558,3.911211,0.485018,0.550626,-0.69,-1.0,52.9
1888,2026-03-13,327.46,333.512,128.969,128.394,346.622,283.223,3.4,2.582,2.354,...,0.320,-5.800000e-02,2.416,2.496,3.911211,0.485018,0.550626,-0.69,-1.0,52.9
1889,2026-03-20,327.46,333.512,128.969,128.394,346.622,283.223,3.4,2.620,2.376,...,0.272,-6.600000e-02,2.426,2.492,3.911211,0.485018,0.550626,-0.69,-1.0,52.9
1890,2026-03-27,327.46,333.512,128.969,128.394,346.622,283.223,3.4,2.542,2.324,...,0.362,1.387779e-18,2.420,2.568,3.911211,0.485018,0.550626,-0.69,-1.0,52.9


### Isolate Features

In [3]:
# ===== FEATURE ISOLATION =====
KEEP = [
    "date",
    # ── Primary model (Phase 2b) ──────────────────────
    "CPI_YoY",                          # DV
    "M2_YoY_Lag_78w",                   # IV: M2 passthrough (core)
    "CPI_Energy_YoY",                   # IV: supply shock control
    "Inflation_Momentum",               # IV: 3m vs 12m timing
    "Expect_Anchor_Signed",             # IV: expectations channel
    # ── Regime split ──────────────────────────────────
    "Debt_to_GDP",                       # threshold variable
    # ── Reaction function (Phase 2c) ──────────────────
    "FedFunds_Rate",                     # builds FFR_Change_13w
    "Taylor_Gap",                        # IV
    "Mandate_Regime",                    # IV: disambiguates mandate quadrants
    "Policy_Stance_SEP",                 # IV
    # ── Scenario calibration (Phase 4) ────────────────
    "M2_YoY",                            # current M2 for scenario design
    "Core_PCE_YoY",                      # alt DV robustness
    # ── Robustness (Phase 6) ──────────────────────────
    "Unemployment_Rate",                 # borderline control
    "Home_Price_YoY_Lag_78w",            # borderline control (housing)
    "Core_CPI_YoY",                      # alt DV
    # ── Sensitivity / decomposition (Phase 5) ─────────
    "Real_FFR_PCE",                      # real rate context
    "Chicago_Fed_Financial_Conditions",  # FCI contradiction
    "UMich_Inflation_Expectations",      # expectations decomposition
    "Breakeven_Inflation_5Y",            # expectations decomposition
    "Breakeven_Inflation_10Y",           # expectations decomposition
    "JOLTS_UE_Ratio",                    # labor market context
]
fred.data = fred.data[KEEP].reset_index(drop=True)
# =============================

### Calculate Debt Regimes (High | Low Percentile)

In [4]:
thresh = fred.data["Debt_to_GDP"].quantile(DEBT_TO_GDP_PERCENTILE/100)
print(f"Regime threshold is {DEBT_TO_GDP_PERCENTILE}th percentile of Debt_to_GDP: {thresh:.2f}")
fred.data["Regime"] = fred.data["Debt_to_GDP"].apply(lambda x: "High" if x > thresh else "Low")
high = Pond(fred.data[fred.data["Regime"]=="High"])
low = Pond(fred.data[fred.data["Regime"]=="Low"])
print(f"High regime has {len(high.data)} observations, Low regime has {len(low.data)} observations.")

print("Date Range:")
print(f"High regime: {high.data['date'].min().date()} to {high.data['date'].max().date()}")
print(f"Low regime: {low.data['date'].min().date()} to {low.data['date'].max().date()}")

Regime threshold is 67th percentile of Debt_to_GDP: 100.11
High regime has 613 observations, Low regime has 1278 observations.
Date Range:
High regime: 2013-01-04 to 2026-03-27
Low regime: 1990-01-05 to 2015-09-25


### Assign Model Vars

In [5]:
specs = []
for label, regime in [("Low", low), ("High", high)]:
    
    regime.clear_caches()
    regime.set_dependent("CPI_YoY")
    regime.add_independents("M2_YoY_Lag_78w","CPI_Energy_YoY","Inflation_Momentum","Expect_Anchor_Signed")
    spec = regime.get_spec()
    specs.append(spec)
    print("============================")
    print(f"{label} Regime Model Spec:")
    print(spec)
    print("============================")

Caches cleared
Dependent variable set to: CPI_YoY
Independent variables: ['M2_YoY_Lag_78w', 'CPI_Energy_YoY', 'Inflation_Momentum', 'Expect_Anchor_Signed']
Low Regime Model Spec:
ModelSpec(n=1278, source=full, dependent=CPI_YoY, independents=['M2_YoY_Lag_78w', 'CPI_Energy_YoY', 'Inflation_Momentum', 'Expect_Anchor_Signed'], controls=[])
Caches cleared
Dependent variable set to: CPI_YoY
Independent variables: ['M2_YoY_Lag_78w', 'CPI_Energy_YoY', 'Inflation_Momentum', 'Expect_Anchor_Signed']
High Regime Model Spec:
ModelSpec(n=613, source=full, dependent=CPI_YoY, independents=['M2_YoY_Lag_78w', 'CPI_Energy_YoY', 'Inflation_Momentum', 'Expect_Anchor_Signed'], controls=[])


In [6]:
results = {}
for label, spec in [("Low", specs[0]), ("High", specs[1])]:
    X = sm.add_constant(spec.X)
    y = spec.y
    
    # Drop rows with NaN in X or y
    mask = X.notna().all(axis=1) & y.notna()
    X_clean = X[mask]
    y_clean = y[mask]
    
    model = sm.OLS(y_clean, X_clean).fit(cov_type="HAC", cov_kwds={"maxlags": 26})
    results[label] = model
    print(f"\n{'='*60}")
    print(f"{label} Debt Regime (n={len(y_clean)})")
    print(f"{'='*60}")
    print(model.summary())


Low Debt Regime (n=1148)
                            OLS Regression Results                            
Dep. Variable:                CPI_YoY   R-squared:                       0.877
Model:                            OLS   Adj. R-squared:                  0.877
Method:                 Least Squares   F-statistic:                     112.0
Date:                Sat, 28 Mar 2026   Prob (F-statistic):           1.34e-80
Time:                        14:20:21   Log-Likelihood:                -568.46
No. Observations:                1148   AIC:                             1147.
Df Residuals:                    1143   BIC:                             1172.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const 

---

# Simulation under different Regimes

In [7]:
betas = {}
for label, model in results.items():
    betas[label] = {
        "const":               model.params["const"],
        "M2_YoY_Lag_78w":      model.params["M2_YoY_Lag_78w"],
        "CPI_Energy_YoY":      model.params["CPI_Energy_YoY"],
        "Inflation_Momentum":  model.params["Inflation_Momentum"],
        "Expect_Anchor_Signed":model.params["Expect_Anchor_Signed"],
        "resid_std":           model.resid.std(),
    }
    print(f"\n{label} regime:")
    for k, v in betas[label].items():
        print(f"  {k:<25s}: {v:+.6f}")


Low regime:
  const                    : +2.512041
  M2_YoY_Lag_78w           : -0.128795
  CPI_Energy_YoY           : +0.083683
  Inflation_Momentum       : -0.171019
  Expect_Anchor_Signed     : +0.354185
  resid_std                : +0.397194

High regime:
  const                    : +1.260336
  M2_YoY_Lag_78w           : +0.103795
  CPI_Energy_YoY           : +0.057989
  Inflation_Momentum       : -0.203434
  Expect_Anchor_Signed     : +1.622718
  resid_std                : +0.718569


---

# Feed Betas to Monte Carlo Model

In [8]:
# ===== MONTE CARLO SIMULATION =====
def make_model(b):
    """
    Build a row-wise model function from a coefficient dict.
    b: beta dict
    """
    def model(row):
        return ( # Solve the OLS
            b["const"]
            + b["M2_YoY_Lag_78w"]       * row["M2_YoY_Lag_78w"]
            + b["CPI_Energy_YoY"]       * row["CPI_Energy_YoY"]
            + b["Inflation_Momentum"]   * row["Inflation_Momentum"]
            + b["Expect_Anchor_Signed"] * row["Expect_Anchor_Signed"]
            + np.random.normal(0, b["resid_std"])
        )
    return model

sims = {}
for label, spec, beta in [("Low", specs[0], betas["Low"]),
                           ("High", specs[1], betas["High"])]:
    sim = Simulation.from_spec(
        spec,
        model=make_model(beta),
        dist_type="normal",
        n_iterations=10_000,
        seed=42,
    )
    sims[label] = sim
    print(f"\n{label} regime simulation ready:")
    print(f"  Variables: {sim.input_manager.variable_names}")
    print(f"  Iterations: {sim.n_iterations}")


# Run both and summarize
for label, sim in sims.items():
    result = sim.run()
    print(f"\n{label} Debt Regime — CPI_YoY distribution:")
    print(f"  Mean:   {result.mean:.2f}%")
    print(f"  Median: {result.median:.2f}%")
    print(f"  Std:    {result.std:.2f}")
    print(f"  95% CI: [{result.ci_lower:.2f}%, {result.ci_upper:.2f}%]")


Low regime simulation ready:
  Variables: ['M2_YoY_Lag_78w', 'CPI_Energy_YoY', 'Inflation_Momentum', 'Expect_Anchor_Signed']
  Iterations: 10000

High regime simulation ready:
  Variables: ['M2_YoY_Lag_78w', 'CPI_Energy_YoY', 'Inflation_Momentum', 'Expect_Anchor_Signed']
  Iterations: 10000

Low Debt Regime — CPI_YoY distribution:
  Mean:   2.44%
  Median: 2.46%
  Std:    1.14
  95% CI: [0.21%, 4.65%]

High Debt Regime — CPI_YoY distribution:
  Mean:   2.87%
  Median: 2.85%
  Std:    2.05
  95% CI: [-1.05%, 6.90%]


# Scenarios:

In [9]:
# ===== SCENARIO COMPARISON =====
scenarios = [
    Scenario("M2_at_4pct", overrides={
        "M2_YoY_Lag_78w": {"mean": 4.0, "std": 1.5},
    }),
    Scenario("M2_at_8pct", overrides={
        "M2_YoY_Lag_78w": {"mean": 8.0, "std": 2.0},
    }),
    Scenario("M2_at_12pct", overrides={
        "M2_YoY_Lag_78w": {"mean": 12.0, "std": 3.0},
    }),
    Scenario("M2_surge_20pct", overrides={
        "M2_YoY_Lag_78w": {"mean": 20.0, "std": 4.0},
    }),
]

for label, sim in sims.items():
    print(f"\n{'='*60}")
    print(f"{label} Debt Regime — Scenario Comparison")
    print(f"{'='*60}")
    summary = sim.compare_scenarios_summary(scenarios)
    print(summary.to_string(index=False, float_format="%.2f"))


Low Debt Regime — Scenario Comparison
      scenario  mean  median  std  ci_lower  ci_upper   min  max
      baseline  2.44    2.44 1.13      0.20      4.67 -2.03 6.70
    M2_at_4pct  2.61    2.60 1.11      0.43      4.78 -1.24 6.79
    M2_at_8pct  2.08    2.08 1.12     -0.10      4.29 -2.02 5.87
   M2_at_12pct  1.57    1.57 1.15     -0.71      3.84 -2.61 5.39
M2_surge_20pct  0.54    0.54 1.20     -1.84      2.90 -3.91 4.70

High Debt Regime — Scenario Comparison
      scenario  mean  median  std  ci_lower  ci_upper   min   max
      baseline  2.88    2.86 2.05     -1.09      6.90 -5.03 10.89
    M2_at_4pct  2.61    2.60 1.72     -0.79      5.97 -4.11  9.76
    M2_at_8pct  3.02    3.02 1.76     -0.37      6.48 -3.83  9.60
   M2_at_12pct  3.42    3.40 1.82     -0.14      6.98 -3.77 10.12
M2_surge_20pct  4.26    4.25 1.87      0.63      7.86 -3.44 10.97


In [10]:
# ===== SENSITIVITY ANALYSIS =====
print("High Debt Regime — Tornado Sensitivity")
print("="*60)
tornado_high = sims["High"].sensitivity.tornado()
print(tornado_high.to_string(index=False, float_format="%.3f"))

print(f"\nLow Debt Regime — Tornado Sensitivity")
print("="*60)
tornado_low = sims["Low"].sensitivity.tornado()
print(tornado_low.to_string(index=False, float_format="%.3f"))

High Debt Regime — Tornado Sensitivity
            variable  low_value  high_value  low_outcome  high_outcome  swing
      CPI_Energy_YoY    -13.271      18.536        1.740         4.573  2.832
      M2_YoY_Lag_78w     -1.532      14.911        1.797         3.776  1.979
  Inflation_Momentum     -0.996       0.990        3.369         2.281  1.088
Expect_Anchor_Signed     -0.270       1.189        1.958         2.789  0.831

Low Debt Regime — Tornado Sensitivity
            variable  low_value  high_value  low_outcome  high_outcome  swing
      CPI_Energy_YoY    -10.056      18.033        1.095         3.498  2.403
      M2_YoY_Lag_78w      2.091       8.471        3.184         1.807  1.376
Expect_Anchor_Signed     -0.038       1.408        2.014         2.267  0.254
  Inflation_Momentum     -0.740       0.652        2.736         2.739  0.003


In [11]:
# ===== VISUALIZATION =====
from otter import SimulationPlotter
plot = SimulationPlotter()

# 1. Scenario comparison — overlaid KDEs for both regimes
high_scenario_results = sims["High"].compare_scenarios(scenarios)
low_scenario_results = sims["Low"].compare_scenarios(scenarios)

# 2. Side-by-side tornado charts
fig = plot.tornado_comparison(
    [tornado_low, tornado_high],
    ["Low Debt", "High Debt"],
    outcome_label="CPI YoY (%)",
    title="Sensitivity Comparison by Debt Regime",
)
fig.show()
fig.write_image("charts/tornado_comparison.png", scale=2)

# 3. Scenario KDE comparison — High debt regime
fig = plot.scenario_comparison(
    high_scenario_results,
    outcome_label="CPI YoY (%)",
    title="High Debt Regime — Scenario Comparison",
)
fig.show()
fig.write_image("charts/high_debt_scenarios.png", scale=2)

# 4. Scenario KDE comparison — Low debt regime
fig = plot.scenario_comparison(
    low_scenario_results,
    outcome_label="CPI YoY (%)",
    title="Low Debt Regime — Scenario Comparison",
)
fig.show()
fig.write_image("charts/low_debt_scenarios.png", scale=2)

# 5. Convergence check
high_baseline = high_scenario_results["baseline"]
fig = plot.convergence_plot(
    high_baseline.outcomes,
    title="High Debt Regime — Convergence",
)
fig.show()
fig.write_image("charts/convergence_high.png", scale=2)

# 5. Convergence check
low_baseline = low_scenario_results["baseline"]
fig = plot.convergence_plot(
    low_baseline.outcomes,
    title="Low Debt Regime — Convergence",
)
fig.show()
fig.write_image("charts/convergence_low.png", scale=2)

# 6. Convergence diagnostics
for label, sim in sims.items():
    result = sim.run()
    conv = sim.check_convergence(result)
    print(f"\n{label} regime convergence:")
    for k, v in conv.items():
        print(f"  {k}: {v}")


Low regime convergence:
  is_converged: True
  suggested_n: 32684
  current_n: 10000
  relative_se: 0.004611978137158937

High regime convergence:
  is_converged: False
  suggested_n: 78815
  current_n: 10000
  relative_se: 0.007161861269186437


# Fed Reaction Analysis

In [12]:
# ===== PHASE 2c: FED REACTION FUNCTION =====

# Construct the dependent variable: 13-week forward change in FFR
fred.data["FFR_Change_13w"] = fred.data["FedFunds_Rate"].shift(-13) - fred.data["FedFunds_Rate"]

# Separate true stagflation from goldilocks overshoot; their shared
# Mandate_Tension product is ambiguous.
fred.data["Mandate_Stagflation"] = (fred.data["Mandate_Regime"] == 3).astype(int)
fred.data["Mandate_Goldilocks_Overrun"] = (fred.data["Mandate_Regime"] == -3).astype(int)

# Estimate on full sample (pooled across regimes)
# The question: what predicts Fed rate moves?
reaction_cols = ["Taylor_Gap", "Inflation_Momentum", "Expect_Anchor_Signed", "Mandate_Stagflation", "Mandate_Goldilocks_Overrun"]
reaction_df = fred.data.dropna(subset=reaction_cols + ["FFR_Change_13w"]).copy()

X_react = sm.add_constant(reaction_df[reaction_cols])
y_react = reaction_df["FFR_Change_13w"]

# Newey-West with bandwidth >= 13 (overlapping 13-week windows)
reaction_model = sm.OLS(y_react, X_react).fit(cov_type="HAC", cov_kwds={"maxlags": 26})
print("Full Sample Reaction Function")
print("="*60)
print(reaction_model.summary())

# Now estimate separately by regime
print("\n\n")
reaction_results = {}
for label in ["Low", "High"]:
    sub = reaction_df[reaction_df["Regime"] == label]
    X = sm.add_constant(sub[reaction_cols])
    y = sub["FFR_Change_13w"]
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 26})
    reaction_results[label] = model
    print(f"\n{'='*60}")
    print(f"{label} Debt Regime — Reaction Function (n={len(y)})")
    print(f"{'='*60}")
    print(model.summary())

Full Sample Reaction Function
                            OLS Regression Results                            
Dep. Variable:         FFR_Change_13w   R-squared:                       0.179
Model:                            OLS   Adj. R-squared:                  0.178
Method:                 Least Squares   F-statistic:                     6.263
Date:                Sat, 28 Mar 2026   Prob (F-statistic):           5.28e-05
Time:                        14:20:34   Log-Likelihood:                -970.27
No. Observations:                1826   AIC:                             1951.
Df Residuals:                    1821   BIC:                             1978.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
co

In [13]:
# Extract reaction function coefficients
reaction_betas = {}
for label, model in reaction_results.items():
    reaction_betas[label] = {
        "const":                 model.params["const"],
        "Taylor_Gap":            model.params["Taylor_Gap"],
        "Inflation_Momentum":    model.params["Inflation_Momentum"],
        "Expect_Anchor_Signed":  model.params["Expect_Anchor_Signed"],
        "Mandate_Stagflation":   model.params["Mandate_Stagflation"],
        "Mandate_Goldilocks_Overrun": model.params["Mandate_Goldilocks_Overrun"],
        "resid_std":             model.resid.std(),
    }
    print(f"\n{label} regime reaction function:")
    for k, v in reaction_betas[label].items():
        print(f"  {k:<25s}: {v:+.6f}")


Low regime reaction function:
  const                    : -0.151377
  Taylor_Gap               : +0.024336
  Inflation_Momentum       : -0.002137
  Expect_Anchor_Signed     : +0.181623
  Mandate_Tension          : -0.055240
  resid_std                : +0.441008

High regime reaction function:
  const                    : -0.213041
  Taylor_Gap               : +0.121081
  Inflation_Momentum       : -0.054717
  Expect_Anchor_Signed     : +0.160600
  Mandate_Tension          : -0.100670
  resid_std                : +0.312416


In [14]:
# ===== RESULTS SUMMARY =====
print("=" * 70)
print("REGIME-CONDITIONAL ESTIMATION SUMMARY")
print("=" * 70)

print("\n1. INFLATION MODEL (CPI_YoY)")
print("-" * 50)
for label in ["Low", "High"]:
    b = betas[label]
    print(f"\n  {label} Debt Regime:")
    print(f"    M2 passthrough:     {b['M2_YoY_Lag_78w']:+.3f}")
    print(f"    Energy passthrough: {b['CPI_Energy_YoY']:+.3f}")
    print(f"    Expectations:       {b['Expect_Anchor_Signed']:+.3f}")
    print(f"    Residual std:       {b['resid_std']:.3f}")

print(f"\n  M2 coefficient swing:          {betas['High']['M2_YoY_Lag_78w'] - betas['Low']['M2_YoY_Lag_78w']:+.3f}")
print(f"  Expectations amplification:    {betas['High']['Expect_Anchor_Signed'] / betas['Low']['Expect_Anchor_Signed']:.1f}×")

print("\n\n2. FED REACTION FUNCTION (FFR_Change_13w)")
print("-" * 50)
for label in ["Low", "High"]:
    b = reaction_betas[label]
    r2 = reaction_results[label].rsquared
    print(f"\n  {label} Debt Regime (R²={r2:.3f}):")
    print(f"    Taylor Gap response:    {b['Taylor_Gap']:+.3f}")
    print(f"    Expectations response:  {b['Expect_Anchor_Signed']:+.3f}")
    print(f"    Stagflation response:  {b['Mandate_Stagflation']:+.3f}")
    print(f"    Goldilocks response:   {b['Mandate_Goldilocks_Overrun']:+.3f}")

print(f"\n  Taylor responsiveness increase: {reaction_betas['High']['Taylor_Gap'] / reaction_betas['Low']['Taylor_Gap']:.1f}×")
print(f"  R² jump: {reaction_results['Low'].rsquared:.3f} → {reaction_results['High'].rsquared:.3f}")

print("\n\n3. SCENARIO HEADLINE (M2 surge = 20% YoY)")
print("-" * 50)
print(f"  High debt regime CPI:  4.87%  (95% CI: [1.26%, 8.55%])")
print(f"  Low debt regime CPI:   0.54%  (95% CI: [-1.88%, 2.94%])")
print(f"  Fiscal amplifier gap:  4.33pp")

REGIME-CONDITIONAL ESTIMATION SUMMARY

1. INFLATION MODEL (CPI_YoY)
--------------------------------------------------

  Low Debt Regime:
    M2 passthrough:     -0.129
    Energy passthrough: +0.084
    Expectations:       +0.354
    Residual std:       0.397

  High Debt Regime:
    M2 passthrough:     +0.104
    Energy passthrough: +0.058
    Expectations:       +1.623
    Residual std:       0.719

  M2 coefficient swing:          +0.233
  Expectations amplification:    4.6×


2. FED REACTION FUNCTION (FFR_Change_13w)
--------------------------------------------------

  Low Debt Regime (R²=0.045):
    Taylor Gap response:    +0.024
    Expectations response:  +0.182
    Mandate Tension:        -0.055

  High Debt Regime (R²=0.511):
    Taylor Gap response:    +0.121
    Expectations response:  +0.161
    Mandate Tension:        -0.101

  Taylor responsiveness increase: 5.0×
  R² jump: 0.045 → 0.511


3. SCENARIO HEADLINE (M2 surge = 20% YoY)
--------------------------------------